# Affine Coupling Flow

*Hard · from [mlcode](https://mlcode-io.vercel.app/problems/torch-coupling-flow)*

Build a RealNVP style normalising flow and train it by maximum likelihood on
two moons.

A flow is an invertible map from data `x` to a latent `z` that has a
standard normal distribution. The change of variables formula gives an
exact log likelihood:

```
log p(x) = log N(z; 0, I) + log |det dz/dx|
```

An **affine coupling layer** makes the determinant cheap. A fixed binary
`mask` picks the coordinates that pass through unchanged. A network looks
only at those coordinates and outputs a log scale `s` and a shift `t` for
the others:

```
x_a = x * mask
s, t = net(x_a)                      # both zero where mask == 1
y = x_a + (1 - mask) * (x * exp(s) + t)
log_det = sum of s over the coordinates
```

The Jacobian is triangular, so its log determinant is the sum of `s`. The
inverse uses the same network, because `x_a = y * mask`:

```
x = y_a + (1 - mask) * (y - t) * exp(-s)
```

Stack layers with alternating masks into a `Flow`. Then train it in
`train_flow` by minimising `-log_prob(X).mean()`.

### Function signature

```python
class AffineCoupling(nn.Module):
    def __init__(self, mask, hidden=64): ...
    def forward(self, x): ...     # -> (y, log_det), log_det of shape (n,)
    def inverse(self, y): ...     # -> x


class Flow(nn.Module):
    def __init__(self, dim=2, n_layers=6, hidden=64): ...
    def forward(self, x): ...     # data to latent -> (z, log_det)
    def inverse(self, z): ...     # latent to data
    def log_prob(self, x): ...    # -> (n,)
    def sample(self, n): ...      # -> (n, dim)


def train_flow(X):
```

### Example

**Input**

```
mask = torch.tensor([1.0, 0.0])
x    = torch.tensor([[2.0, 3.0]])
# suppose net(x * mask) gives s = 0.5 and t = 1.0 for coordinate 1
```

**Output**

```
y       = tensor([[2.0000, 5.9462]])
log_det = tensor([0.5000])
```

**Reasoning**

Coordinate 0 has mask `1`, so it passes through as `2.0`. Coordinate 1 is
scaled and shifted: `3 * exp(0.5) + 1 = 5.9462`. Only one coordinate is
scaled, so the log determinant is that coordinate's `s`, which is `0.5`.

### Constraints and notes

- `mask` is a float tensor of zeros and ones with shape `(dim,)`. The
  coordinates where it is `1` must come out unchanged.
- The tests compare your `log_det` with the log determinant of the true
  Jacobian, which they get with `torch.autograd.functional.jacobian`, after
  they set your weights to random values. Your layers must work in float64.
- `inverse(forward(x))` must give back `x`, for one layer and for the flow.
- Use alternating masks in `Flow`, so every coordinate gets transformed.
- `train_flow` gets a float tensor of shape `(n, 2)`. On held-out points
  from the same distribution, the mean log likelihood must be above `-2.3`.
  The best single Gaussian gets about `-2.7`.
- Keep training under about 10 seconds on a CPU.

---

Write your solution in the cell below, then run the grading cell at the bottom. Your work is recorded against the same account as the site, so a pass here shows there.

Keep the solution in the function you were given. The grading cell saves your definitions and the lines around them back to the site, not the whole session, so anything you type to poke about is left behind rather than ending up in your saved answer.

In [ ]:
#@title Connect this notebook to mlcode { display-mode: "form" }
#@markdown Paste the code from the problem page, then run this cell.
CODE = ""  #@param {type:"string"}

SITE = "https://mlcode-io.vercel.app"
SLUG = "torch-coupling-flow"

import json, urllib.request, urllib.error

def _mlc_get(path):
    with urllib.request.urlopen(SITE + path, timeout=30) as r:
        return r.read()

# The harness is fetched rather than pasted in, so the cases run here exactly
# as they would on the site.
for _mlc_n in ("harness.py", "notebook.py"):
    with open("/content/" + _mlc_n, "wb") as _mlc_fh:
        _mlc_fh.write(_mlc_get("/" + _mlc_n))

try:
    _mlc_meta = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))
    print("connected:", _mlc_meta["title"])
    print("torch:", __import__("torch").__version__)
except urllib.error.HTTPError as e:
    print("could not connect:", json.loads(e.read()).get("error", e.reason))
    print("Get a fresh code from the problem page; they last three hours.")


In [ ]:
import torch
from torch import nn
from torch.distributions import Normal


class AffineCoupling(nn.Module):
    """One affine coupling layer. Coordinates where mask == 1 pass through."""

    def __init__(self, mask, hidden=64):
        pass

    def forward(self, x):
        """x: (n, dim). Returns (y, log_det) with log_det of shape (n,)."""
        pass

    def inverse(self, y):
        """The exact inverse of forward."""
        pass


class Flow(nn.Module):
    """A stack of coupling layers with alternating masks, over N(0, I)."""

    def __init__(self, dim=2, n_layers=6, hidden=64):
        pass

    def forward(self, x):
        """Data to latent. Returns (z, total log_det)."""
        pass

    def inverse(self, z):
        """Latent to data."""
        pass

    def log_prob(self, x):
        """Exact log density of each row of x, shape (n,)."""
        pass

    def sample(self, n):
        """Draw n points from the model, shape (n, dim)."""
        pass


def train_flow(X):
    """Fit a Flow to X of shape (n, 2) by maximum likelihood and return it."""
    pass

In [ ]:
#@title Grade this notebook { display-mode: "form" }
#@markdown Runs the hidden tests and records the attempt on the site.
import json, sys, time, urllib.request, urllib.error

sys.path.insert(0, "/content")
import harness

# The tests are fetched now rather than stored in the notebook, so a reader
# cannot read them before attempting the problem.
_mlc_tests = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))["tests"]

# `In` is the execution history, not the current cells: a wrong first attempt,
# a cell that errored and three re-runs are all still in it. Saving that would
# put something on the site that does not even replay. Rebuild the source from
# what is actually defined now instead, and keep the history only as a fallback
# for a solution written at module level.
def _mlc_collect():
    import ast, inspect, types

    # This notebook's own form cells are not part of anyone's answer.
    _cells = [c for c in In[1:] if not c.lstrip().startswith("#@title")]

    _imports, _seen = [], set()
    _consts = dict()
    for _cell in _cells:
        for _line in _cell.split("\n"):
            _t = _line.strip()
            if (_t.startswith("import ") or _t.startswith("from ")) and _t not in _seen:
                _seen.add(_t)
                _imports.append(_t)
        try:
            _tree = ast.parse(_cell)
        except SyntaxError:
            continue
        # Module level constants the functions below may close over. Last
        # assignment wins, because an edited cell is run again.
        for _node in _tree.body:
            if isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                _target = _node.targets[0]
                if isinstance(_target, ast.Name):
                    _text = ast.get_source_segment(_cell, _node)
                    if _text and len(_text) <= 300:
                        # A plain literal can go above the definitions, where a
                        # default argument can still see it. Anything else may
                        # call one of them, so it has to come after.
                        try:
                            ast.literal_eval(_node.value)
                            _lit = True
                        except Exception:
                            _lit = False
                        _consts[_target.id] = (_text, _lit)

    _defs, _defined = [], set()
    for _name, _obj in list(globals().items()):
        if _name.startswith("_mlc_") or _name in ("In", "Out"):
            continue
        if isinstance(_obj, (types.FunctionType, type)) and getattr(_obj, "__module__", "") == "__main__":
            try:
                _lead = inspect.getcomments(_obj) or ""
                _defs.append(_lead + inspect.getsource(_obj).rstrip())
                _defined.add(_name)
            except (OSError, TypeError):
                pass

    # Nothing defined means the solution is not shaped like a function, so fall
    # back to what was typed rather than returning an empty file.
    if not _defs:
        return "\n\n".join(_cells)

    _early, _late = [], []
    for _name, _pair in _consts.items():
        if _name in _defined or _name.startswith("_mlc_") or _name in ("CODE", "SITE", "SLUG"):
            continue
        # What is kept is the line of source, not the value it produced, so how
        # large the value is does not matter. The length bound above is what
        # keeps a pasted-in data literal out.
        (_early if _pair[1] else _late).append(_pair[0])

    _blocks = ["\n".join(_imports), "\n".join(_early),
               "\n\n\n".join(_defs), "\n".join(_late)]
    return "\n\n\n".join(b for b in _blocks if b.strip()) + "\n"

_mlc_source = _mlc_collect()

_mlc_started = time.time()
_mlc_results = harness.run_tests(_mlc_tests, dict(globals()), 60, _mlc_source, "submit")
_mlc_elapsed = int((time.time() - _mlc_started) * 1000)

_mlc_passed = sum(1 for r in _mlc_results if r["passed"])
for _mlc_r in _mlc_results:
    print(("  pass  " if _mlc_r["passed"] else "  FAIL  ") + _mlc_r["name"])
    if not _mlc_r["passed"] and _mlc_r["message"]:
        print("        " + _mlc_r["message"].replace("\n", "\n        "))
print(f"\n{_mlc_passed}/{len(_mlc_results)} passed")

_mlc_body = json.dumps({
    "code": CODE, "slug": SLUG, "passedCount": _mlc_passed,
    "totalCount": len(_mlc_results), "runtimeMs": _mlc_elapsed, "source": _mlc_source,
}).encode()
_mlc_req = urllib.request.Request(SITE + "/api/colab", data=_mlc_body,
                              headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(_mlc_req, timeout=30) as r:
        print("recorded on mlcode:", "solved" if json.load(r)["passed"] else "attempt saved")
except urllib.error.HTTPError as e:
    print("could not record:", json.loads(e.read()).get("error", e.reason))
